In [ ]:
# (종합) 네이버 블로그에서 [서울 버거 맛집] 웹크로링 데이터 약 200개 - 전처리

import pandas as pd
import re
import csv
from konlpy.tag import Okt


# csv 로 저장된 파일 열기
df = pd.read_csv('서울_버거_맛집_200.csv')

# title 컬럼에 대한 데이터 전처리 ( re, pandas )

# 가 정규식을 사용해서 데이터 전처리
# 가-1. 정규식을 사용해서 특수문자 제거 (한글, 영어, 숫자, 공백 제외)
df['title'] = df['title'].str.replace(r'[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z0-9 ]', '', regex=True)
# 가-2. 정규식으로 연도, 월 정보 제거 
df['title'] = df['title'].str.replace(r'\d{4}년?\s*\d{1,2}월?', '', regex=True)
df['title'] = df['title'].str.replace(r'\b\d{4}[.\s]?\d{1,2}\b\s?', '', regex=True)

# 나 표준화
# 나-1. 영문자 대문자 -> 소문자로
df['title'] = df['title'].str.lower()
# df['title'] = df['title'].str.replace('[a-zA-Z]', '', regex=True) # 옵션 : 영문자 제외

# 다 데이터 가공
# 다-1 title, content 에 데이터가 없으면 삭제
df = df.dropna(subset=['title', 'content'])

# 다-2 title 에 데이터가 중복이 경우
df = df.drop_duplicates(subset=['title'])

# 라 기타 데이터 제거
# title 컬럼에 제주, 부산 있는지 확인? 과천, 제주, 부산, 괌 인경우 행을 삭제
# df[df['title'].str.contains('제주')]
# 라-1 서울이 아닌 지역이 있으면 행 제거
df = df[~df['title'].str.contains('과천|제주|부산|괌|양평', na=False)]

# 라-2 불용어 (STOPWORD) 를 제거
stopwords = ["에서", "까지", "와", "과", "의", "로","이","가","는", "보다"] # 샘플로 10개만 
pattern = "|".join(stopwords)
df["title"] = df["title"].str.replace(pattern, "", regex=True)

# 라-3 연속된 공백제거하여 한개 공백
df["title"] = df["title"].str.replace(r'\s+', ' ', regex=True)

#df.iloc[0:50]
#print(len(df)) # 203 -> 202(중복) -> 다른지역(5개-197)

# content 컬럼에 대한 데이터 전처리 ( re, pandas, konlpy )

def clean_blog_content(text) :
    # 결측치(NaN) 처리
    if not isinstance(text, str):
        return ""
    # 가-1 
    # URL 제거 패턴
    url_pattern = r'https?://[a-zA-Z0-9\-.:/?=&_]+'
    # 이메일 제거 패턴
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    # 특수문자 제거 패턴 (한글, 영문, 숫자, 공백만 남김)
    special_pattern = r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]'

    # 순차적으로 정규식 적용
    text = re.sub(url_pattern, '', text)
    text = re.sub(email_pattern, '', text)
    text = re.sub(special_pattern, '', text)

    # 연속된 공백 하나로 줄이기 및 앞뒤 공백 제거
    text = re.sub(r'\s+', ' ', text)

    # 나-1 konlpy를 적용 - 명사 추출
    nouns = okt.nouns(text)
    # '것', '수', '등'과 같은 의미 없는 1글자 불용어 제거
    # 나-2 제거하고 싶은 불용어 리스트 정의
    stop_words = ['정말', '진짜', '오늘', '이번', '로그', '블로그', '네이버', '대박', '사건']
    nouns = [noun for noun in nouns if (noun not in stop_words) and (len(noun) > 1 or noun == '맛')]
    
    return " ".join(nouns)

#  clean_blog_content 함수를 df 적용
df['content_clean'] = df['content'].apply(clean_blog_content)

#df.iloc[0:50]
# 데이터 중간 저장
output_file = '서울_버거_맛집_200_전처리.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print('CSV_전처리 파일 저장완료')




In [11]:
# csv로 저장된 파일 열기

import pandas as pd

df= pd.read_csv('탈모검색_500.csv')

#df.head(50)

In [12]:
# 가-1. 'title' 컬럼의 특수문자 제거 (정규식 활용)
    # [^a-zA-Z0-9가-힣 ] : 영어, 숫자, 한글, 공백이 아닌 모든 것을 찾음

df['title'] = df['title'].str.replace(r'[^a-zA-Z0-9가-힣 ]', '', regex=True)

df.head(10)

,title,link,date,content
0,다발성 원형탈모증 증상과 치료법 방치하면 어떻게 될까,https://blog.naver.com/y_nio/224305465047,2026. 6. 4.,"다발성 원형탈모증 증상과 치료법, 방치하면 어떻게 될까?""어? 머리카락이 또 빠졌네..."
1,피나스테리드 두타스테리드 차이 성분 효과 기전 탈모약 부작용,https://blog.naver.com/kingbed4/224311950353,2026. 6. 10.,피나스테리드 두타스테리드 차이 성분 효과 기전 탈모약 부작용​안녕하세요 약사의약설명...
2,출산후 탈모 언제까지 빠지는 이유와 관리법 완벽 정리,https://blog.naver.com/y_nio/224259750634,2026. 4. 23.,출산후 탈모 언제까지? 빠지는 이유와 관리법 완벽 정리아이를 낳고 나서 기쁨도 잠시...
3,견인성 탈모 원인 증상과 피부과 실비 보험 보상 기준 알아보기,https://blog.naver.com/sealom2/224288144383,2026. 5. 22.,견인성 탈모 원인 증상과 피부과 실비 보험 보상 기준 알아보기​견인성 탈모는 두피와...
4,탈모치료약 사용법,https://blog.naver.com/afseol/224305550285,2026. 6. 4.,"탈모치료약 사용법탈모치료약은 크게​먹는 약,바르는 약,병원에서 진행하는 치료제로 나..."
5,탈모에 좋은 음식 머리카락 빠질 때 도움되는 영양 성분 정리,https://blog.naver.com/njellsol1/224294425010,2026. 5. 25.,"탈모에 좋은 음식, 머리카락 빠질 때 도움되는 영양 성분 정리머리를 감을 때 빠지는..."
6,원형탈모원인 대표적인 것들,https://blog.naver.com/cera23/224287127396,2026. 5. 16.,"원형탈모원인 대표적인 것들​원형탈모 원인, 대표적인 것들원형탈모는 갑자기 동전처럼 ..."
7,그냥 머리가 좀 가려웠는데 단 열흘 만에 겪은 초고속 탈모의 진실,https://blog.naver.com/awsomi/224317330955,2026. 6. 16.,"""그냥 머리가 좀 가려웠는데.."" 단 열흘 만에 겪은 초고속 탈모의 진실단 열흘만에..."
8,바르는탈모약 너무 믿지마세요,https://blog.naver.com/cera23/224287127049,2026. 5. 16.,"바르는탈모약 너무 믿지마세요​바르는 탈모약, 너무 믿지 마세요탈모 관리에서 바르는 ..."
9,탈모클리닉 도움될까 비용은,https://blog.naver.com/cera23/224287084498,2026. 5. 16.,탈모클리닉 도움될까? 비용은?​탈모 클리닉 도움될까? 비용까지 현실 정리​탈모가 시...


In [13]:
# 가-2. 연도/월 등 날짜가 있는 경우 제거

df['title'] = df['title'].str.replace(r'\d{2,4}[-./]\d{1,2}', '', regex=True)
df['title'] = df['title'].str.replace(r'\d{4}년\s?\d{1,2}월', '', regex=True)

In [14]:
# 나-1. 영문자를 모두 소문자로 변경

# 'title' 컬럼의 모든 대문자를 소문자로 변환
df['title'] = df['title'].str.lower()

In [15]:
# (option) 영문자 모두 제거하고 싶은 경우

# 1. 영문자 제거
df['title'] = df['title'].str.replace(r'[a-zA-Z]', '', regex=True)

# 2. 제거 후 남은 불필요한 앞뒤 공백 제거
df['title'] = df['title'].str.strip()

# 3. 만약 중간에 남은 다중 공백을 하나로 줄이고 싶다면
df['title'] = df['title'].str.replace(r'\s+', ' ', regex=True)

In [16]:
# 'title' 또는 'content' 컬럼에 NaN이 있는 행 삭제
df.dropna(subset=['title', 'content'], how='any', inplace=True)

In [17]:
# 삭제된 데이터(행)가 몇 개인지 알고 싶을 때

print("삭제 전 행 개수:", len(df))
df.dropna(subset=['title', 'content'], how='all', inplace=True)
print("삭제 후 행 개수:", len(df))

삭제 전 행 개수: 687
삭제 후 행 개수: 687


In [18]:
# 다-2. title 데이터가 중복인 경우

# 'title' 컬럼을 기준으로 중복된 행을 삭제 (첫 번째 항목만 남기고 나머지는 삭제)
df.drop_duplicates(subset=['title'], inplace=True)
print("삭제 후 행 개수:", len(df))

삭제 후 행 개수: 591


In [19]:
# 데이터 중간 저장

output_file = '탈모검색_500_전처리_1.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print('CSV_전처리_1 파일 저장 완료')

CSV_전처리_1 파일 저장 완료
